# Hierarchical Semantics Indexing

This notebook implements **hierarchical semantics indexing** for GRAM, which assigns each item a hierarchical lexical ID based on its semantic content.

**Prerequisites:** Run `0_nvembed_extraction.ipynb` first to extract item embeddings.

**Pipeline:**
1. Load item data and pre-computed NV-Embed embeddings
2. Apply hierarchical k-means clustering on item embeddings
3. Compute TF-IDF scores for each cluster using T5 vocabulary
4. Generate unique hierarchical lexical IDs by selecting representative tokens per cluster

**Output:** `item_generative_indexing_hierarchy_v1_c{n_clusters}_l{l}.txt`

**Requirements:**
```bash
pip install torch transformers numpy scipy scikit-learn tqdm
```

In [ ]:
import os
import pickle
import warnings
import numpy as np
from collections import defaultdict, Counter
from time import time
from tqdm import tqdm
from scipy.sparse import csc_matrix

warnings.filterwarnings('ignore')


def read_lines(path):
    with open(path, 'r') as f:
        return [line.rstrip('\n') for line in f]


def get_dict_from_lines(lines):
    index_map = {}
    for line in lines:
        info = line.split(' ', 1)
        if len(info) == 2:
            index_map[info[0]] = info[1]
    return index_map

## Configuration

Set dataset name and hyperparameters here.

In [ ]:
# ===== Dataset Configuration =====
dataset_name = "Beauty"          # Options: Beauty, Sports, Toys, Yelp
data_path = "../rec_datasets"    # Path to rec_datasets directory
embedding_dir = "./embeddings"   # Directory where NV-Embed embeddings are stored (see 0_nvembed_extraction.ipynb)

# ===== Embedding Configuration =====
max_length = 4096   # Must match the max_length used in 0_nvembed_extraction.ipynb

# ===== Clustering Hyperparameters =====
n_clusters = 128    # Number of clusters per level (tuned among {32, 128, 512})
l = 7               # Number of hierarchy levels (tuned among {5, 7, 9})
seed = 42           # Random seed

# ===== TF-IDF Hyperparameters =====
tf_max_len = 128    # Maximum token length for TF-IDF computation
topk_tokens = 50    # Number of candidate tokens per cluster

print(f"Dataset: {dataset_name}")
print(f"Clustering: n_clusters={n_clusters}, l={l}, seed={seed}")

## Step 1: Load Item Data and Embeddings

In [ ]:
item_text_file = os.path.join(data_path, dataset_name, "item_plain_text.txt")
item2input = get_dict_from_lines(read_lines(item_text_file))
item_id_list = list(item2input.keys())
item_text_list = list(item2input.values())
asin2idx = {asin: idx for idx, asin in enumerate(item_id_list)}

print(f"Loaded {len(item2input)} items from {dataset_name}")
print(f"Sample: {item_id_list[0]}")
print(f"  Text: {item_text_list[0][:150]}...")

# Load pre-computed NV-Embed embeddings (generated by 0_nvembed_extraction.ipynb)
embedding_file = os.path.join(embedding_dir, f"{dataset_name.lower()}_nvembed_len{max_length}.npy")
if not os.path.exists(embedding_file):
    raise FileNotFoundError(
        f"Embedding file not found: {embedding_file}\n"
        "Please run 0_nvembed_extraction.ipynb first."
    )
item_embeddings = np.load(embedding_file)
print(f"Loaded embeddings: {item_embeddings.shape}")

## Step 2: Hierarchical K-means Clustering

We apply hierarchical k-means clustering on item embeddings to build a `l`-level cluster tree. Items within the same cluster share an identical prefix in their hierarchical ID, capturing semantic relationships.

The cluster mapping is saved to disk and reloaded automatically if it already exists.

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans

cluster_mapping_file = os.path.join(
    embedding_dir,
    f"{dataset_name.lower()}_cluster_mapping_c{n_clusters}_l{l}_seed{seed}.pkl"
)

if os.path.exists(cluster_mapping_file):
    print(f"Loading existing cluster mapping from {cluster_mapping_file}")
    with open(cluster_mapping_file, 'rb') as f:
        mapping = pickle.load(f)
else:
    print(f"Running hierarchical k-means (n_clusters={n_clusters}, l={l}, seed={seed})...")

    new_id_list = [[] for _ in range(len(item_id_list))]
    kmeans = KMeans(
        n_clusters=n_clusters, max_iter=300, n_init=100,
        init='k-means++', random_state=seed, tol=1e-7
    )
    mini_kmeans = MiniBatchKMeans(
        n_clusters=n_clusters, max_iter=300, n_init=100,
        init='k-means++', random_state=seed,
        batch_size=1000, reassignment_ratio=0.01,
        max_no_improvement=20, tol=1e-7
    )

    def classify_recursion(x_data_pos, cur_l):
        if cur_l == l:
            return
        if x_data_pos.shape[0] <= n_clusters:
            for i, pos in enumerate(x_data_pos):
                new_id_list[pos].append(i)
                classify_recursion(np.array([pos]), cur_l + 1)
        else:
            temp_data = item_embeddings[x_data_pos]
            clusterer = mini_kmeans if x_data_pos.shape[0] >= 1000 else kmeans
            pred = clusterer.fit_predict(temp_data)
            for i in range(n_clusters):
                pos_lists = x_data_pos[pred == i]
                for pos in pos_lists:
                    new_id_list[pos].append(i)
                if len(pos_lists) > 0:
                    classify_recursion(pos_lists, cur_l + 1)

    start_time = time()
    print("Step 3-1: Initial clustering...")
    pred = mini_kmeans.fit_predict(item_embeddings)
    for i, class_ in enumerate(pred):
        new_id_list[i].append(class_)

    print("Step 3-2: Recursive clustering...")
    for i in tqdm(range(n_clusters), desc="Processing first-level clusters"):
        pos_lists = np.where(pred == i)[0]
        if len(pos_lists) > 0:
            classify_recursion(pos_lists, cur_l=1)

    mapping = {item_id_list[i]: new_id_list[i] for i in range(len(item_id_list))}

    with open(cluster_mapping_file, 'wb') as f:
        pickle.dump(mapping, f)
    print(f"Clustering complete in {time() - start_time:.1f}s. Saved to {cluster_mapping_file}")

# Build cluster_dict: {cluster_prefix -> [item_asins]}
cluster_dict = defaultdict(list)
for asin, cluster in mapping.items():
    for step in range(len(cluster)):
        prefix = '-'.join(str(c) for c in cluster[:step + 1])
        cluster_dict[prefix].append(asin)

print(f"Total clusters: {len(cluster_dict)}")

## Step 3: TF-IDF Scoring

We compute TF-IDF scores for each item using the T5 vocabulary. These scores are used to identify the most representative tokens for each cluster.

For each cluster, we average the TF-IDF vectors of all items within it and select the highest-scoring token as the cluster's representative.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('t5-base', use_fast=True)
vocab_size = len(tokenizer)

print(f"T5 vocabulary size: {vocab_size}")
print("Building document-term matrix...")

rows, cols, vals = [], [], []
token2df = {}

for i, (asin, text) in tqdm(enumerate(item2input.items()),
                             total=len(item2input), desc="Computing TF-IDF"):
    # Count document frequency using full text
    tokens_full = tokenizer.tokenize(text)
    for token in set(tokens_full):
        token_id = tokenizer.convert_tokens_to_ids(token)
        token2df[token_id] = token2df.get(token_id, 0) + 1

    # Count term frequency using truncated text (max tf_max_len tokens)
    tokens_trunc = tokenizer.tokenize(text)[:tf_max_len]
    token_counter = Counter(tokens_trunc)
    for token, count in token_counter.items():
        token_id = tokenizer.convert_tokens_to_ids(token)
        rows.append(i)
        cols.append(token_id)
        vals.append(count)

doc_matrix_arr = csc_matrix(
    (vals, (rows, cols)), shape=(len(item2input), vocab_size)
).toarray()

# Compute IDF: log(1 + (N - df + 0.5) / (df + 0.5)) (smooth IDF)
n_docs = len(item2input)
idf_vector = np.zeros(vocab_size)
for token_id, df in token2df.items():
    idf_vector[token_id] = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))

# Compute TF-IDF scores per item
term_score_arr = doc_matrix_arr * idf_vector
print(f"TF-IDF matrix shape: {term_score_arr.shape}")

In [ ]:
def filter_tokens(tokens):
    """Remove special tokens and single-character tokens."""
    return [
        t for t in tokens
        if t not in tokenizer.all_special_tokens
        and t not in ['\u2581', '\u2581\u2581']
        and len(str(t)) >= 2
    ]


cid2topk = {}

print("Computing cluster-level TF-IDF scores...")
for cid, asins in tqdm(cluster_dict.items(), desc="Processing clusters"):
    arr_indices = [asin2idx[asin] for asin in asins]

    # Average TF-IDF vectors within cluster (as described in the paper)
    cluster_score = np.mean(term_score_arr[arr_indices], axis=0)

    top_token_ids = cluster_score.argsort()[::-1][:topk_tokens]
    top_tokens = tokenizer.convert_ids_to_tokens(top_token_ids)
    cid2topk[cid] = filter_tokens(top_tokens)

print(f"Cluster token assignment complete: {len(cid2topk)} clusters")

## Step 4: Generate Hierarchical Lexical IDs

For each item, we construct its hierarchical ID by selecting the most representative token at each level of the cluster tree. Tokens are chosen greedily: at each level, the highest-scoring token that has not already been used in a previous level is selected.

Duplicate IDs are resolved by appending a numeric suffix.

In [ ]:
asin2lexid = {}
lexid_cnt = Counter()

for asin in item_id_list:
    cluster = mapping[asin]
    tmp_token = []

    for step in range(len(cluster)):
        prefix = '-'.join(str(c) for c in cluster[:step + 1])
        for token in cid2topk.get(prefix, []):
            if token not in tmp_token:
                tmp_token.append(token)
                break

    lexid = '-'.join(tmp_token)
    asin2lexid[asin] = lexid
    lexid_cnt[lexid] += 1

# Deduplication: append digit suffix for duplicate IDs
asin2lexid_unique = {}
lexid_new_cnt = Counter()

for asin in item_id_list:
    lexid = asin2lexid[asin]
    if lexid_cnt[lexid] == 1:
        asin2lexid_unique[asin] = lexid
    else:
        asin2lexid_unique[asin] = f"{lexid}-{lexid_new_cnt[lexid]}"
        lexid_new_cnt[lexid] += 1

unique_count = len(set(asin2lexid_unique.values()))
total_count = len(asin2lexid_unique)
print(f"Unique IDs: {unique_count} / {total_count} ({unique_count / total_count * 100:.2f}%)")
print("\nSample hierarchical IDs:")
for asin in item_id_list[:5]:
    print(f"  {asin} -> {asin2lexid_unique[asin]}")

## Step 5: Save Output

The output file maps each item ASIN to its hierarchical lexical ID in pipe-separated format:
```
{ASIN} |{token_1}|{token_2}|...|{token_l}
```

For example:
```
B000P24EI2 |▁loss|▁brake|▁fur|▁consolid|ren|▁scalp|▁profound
```

In [ ]:
output_filename = f"item_generative_indexing_hierarchy_v1_c{n_clusters}_l{l}.txt"
output_file = os.path.join(data_path, dataset_name, output_filename)

with open(output_file, 'w') as f:
    for asin in item_id_list:
        hid = asin2lexid_unique[asin]
        tokens = hid.split('-')
        # Format: |token1|token2|...|tokenN
        hid_formatted = '|' + '|'.join(tokens)
        f.write(f"{asin} {hid_formatted}\n")

print(f"Saved to: {output_file}")
print(f"Total items: {len(item_id_list)}")
print("\nSample output:")
for asin in item_id_list[:5]:
    hid = asin2lexid_unique[asin]
    tokens = hid.split('-')
    hid_formatted = '|' + '|'.join(tokens)
    print(f"  {asin} {hid_formatted}")